# Hseq · 02 Silver
Typed and checked. The **quality gate** runs every rule first; if one fails, nothing is written and silver keeps the previous version. Checks against other domains read their silver tables through shortcuts.

In [ ]:
# Domain: hseq · owner: Quality and HSE manager
# Attach this notebook to the lakehouse "lh_hseq" in the workspace "Solvi Hseq".
DOMAIN = "hseq"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to f"{DOMAIN}_"
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named <domain>_<name>
    return spark.read.table(f"{domain}_{name}")

In [ ]:
SCHEMA = {
 "compliance_requirement": {
  "requirement_id": "string",
  "requirement_name": "string",
  "applies_to": "string",
  "due_days": "int",
  "recurring": "int"
 },
 "compliance_item": {
  "item_id": "string",
  "installation_id": "string",
  "requirement_id": "string",
  "due_date": "date",
  "submitted_date": "date"
 },
 "inspection_deviation": {
  "deviation_id": "string",
  "installation_id": "string",
  "found_date": "date",
  "source": "string",
  "category": "string",
  "severity": "string",
  "closed_date": "date"
 },
 "risk_rule": {
  "factor_code": "string",
  "factor": "string",
  "category": "string",
  "weight": "int",
  "stops_work": "int"
 },
 "hse_event": {
  "event_id": "string",
  "job_id": "string",
  "event_date": "date",
  "event_type": "string",
  "severity": "string",
  "description": "string",
  "reported_by": "string"
 },
 "hse_action": {
  "action_id": "string",
  "job_id": "string",
  "action_date": "date",
  "risk_level": "string",
  "action": "string",
  "completed": "int"
 }
}
KEYS = {"compliance_requirement": ["requirement_id"], "compliance_item": ["item_id"], "inspection_deviation": ["deviation_id"], "risk_rule": ["factor_code"], "hse_event": ["event_id"], "hse_action": ["action_id"]}
FKS = [["compliance_item", "installation_id", "installation", "installation_id", "assets"], ["compliance_item", "requirement_id", "compliance_requirement", "requirement_id", "hseq"], ["inspection_deviation", "installation_id", "installation", "installation_id", "assets"], ["hse_event", "job_id", "job", "job_id", "delivery"], ["hse_action", "job_id", "job", "job_id", "delivery"]]      # child, column, parent, column, parent's domain
RANGES = [["inspection_deviation", "severity IN ('High','Medium','Low')"], ["inspection_deviation", "closed_date IS NULL OR closed_date >= found_date"], ["hse_event", "event_type IN ('Incident','Near miss','Good catch')"], ["risk_rule", "weight BETWEEN 1 AND 5"]]

def clean(name):
    df = read_table(f"bronze_{name}")
    return df.select(*[F.nullif(F.trim(F.col(c)), F.lit("")).cast(t).alias(c) for c, t in SCHEMA[name].items()])
silver = {n: clean(n) for n in SCHEMA}
def parent(table, dom):
    return silver[table] if dom == DOMAIN else read_shared(dom, f"silver_{table}")

results = []
for t, keys in KEYS.items():
    df = silver[t]
    results += [(t, "key not null", df.filter(" OR ".join(f"{k} IS NULL" for k in keys)).count()),
                (t, "key unique", df.groupBy(*keys).count().filter("count > 1").count())]
for c, cc, p, pc, dom in FKS:
    orphans = silver[c].filter(F.col(cc).isNotNull()).join(parent(p, dom).select(F.col(pc).alias(cc)), cc, "left_anti").count()
    results.append((c, f"{cc} -> {dom}.{p}", orphans))
for t, cond in RANGES:
    results.append((t, cond, silver[t].filter(f"NOT ({cond})").count()))
dq = (spark.createDataFrame(results, "table string, rule string, failing_rows long")
        .withColumn("passed", F.col("failing_rows") == 0).withColumn("checked_at", F.current_timestamp()))
display(dq.orderBy("passed", "table"))
failed = dq.filter("NOT passed").count()
if failed:
    raise Exception(f"Quality gate failed in {DOMAIN}: {failed} rule(s). Nothing written to silver.")

In [ ]:
for name, df in silver.items():
    write_table(df, f"silver_{name}")
write_table(dq, "silver_dq_log")